In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

import mw_analysis.utils as utils
import mw_analysis.snirf as snirf

from scipy.stats import sem
from snirf import validateSnirf

CHANNELS = [
    "rx1_l1", "rx1_l2", "rx1_l3", "rx1_l4",
    "rx1_l5", "rx1_l6", "rx1_l7", "rx1_l8",
    "rx1_l9", "rx1_l10", "rx1_l11", "rx1_l12",
    "rx1_l13", "rx1_l14", "rx1_l15", "rx1_l16",
    "rx2_l1", "rx2_l2", "rx2_l3", "rx2_l4",
    "rx2_l5", "rx2_l6", "rx2_l7", "rx2_l8",
    "rx2_l9", "rx2_l10", "rx2_l11", "rx2_l12",
    "rx2_l13", "rx2_l14", "rx2_l15", "rx2_l16",
    "battery_voltage", "timestamp"
]

MAPPINGS = {
    "S1_D1": ("rx1_l1", "rx1_l2"),
    "S1_D2": ("rx1_l3", "rx1_l4"),
    "S1_D3": ("rx1_l5", "rx1_l6"),
    "S1_D4": ("rx1_l7", "rx1_l8"),
    "S2_D1": ("rx2_l9", "rx2_l10"),
    "S2_D2": ("rx2_l11", "rx2_l12"),
    "S2_D3": ("rx2_l13", "rx2_l14"),
    "S2_D4": ("rx2_l15", "rx2_l16"),
}

# Processed datasets
datasets = []

for subject_id in range(1, 9):
    stream_df, events_df = utils.nirscord_h5_to_df(f"../data/participant{subject_id}-artinis.h5", CHANNELS)

    # Clean the dataframes
    events_df = utils.clean_events(events_df)

    #
    timestamp = utils.find_experiment_start(events_df)
    stream_df = stream_df.loc[stream_df["timestamp"] >= timestamp]
    events_df = events_df.loc[events_df["timestamp"] >= timestamp]

    #
    stream_df["timestamp"] -= timestamp
    events_df["timestamp"] -= timestamp

    # Convert to HbO/Hb
    hbo_df = stream_df[["timestamp"]]

    for name, (ch1, ch2) in MAPPINGS.items():

        if utils.sci(stream_df[ch1].values, stream_df[ch2].values) < 0.9:
            hbo_df[f"{name} hbo"] = np.full(len(stream_df), np.nan)
            hbo_df[f"{name} hb"] = np.full(len(stream_df), np.nan)
        else:
            HbO, Hb = utils.mbll(stream_df[ch1].values, stream_df[ch2].values)
            hbo_df[f"{name} hbo"] = utils.iir_filter(HbO)
            hbo_df[f"{name} hb"] = utils.iir_filter(Hb)

    # Extract short channel regressor columns
    hbo_cols = hbo_df.filter(regex="hbo$").columns.tolist()
    hb_cols = hbo_df.filter(regex="hb$").columns.tolist()

    # Apply spike detection and removal
    hbo_df = utils.detect_motion_spikes(hbo_df, hbo_cols + hb_cols)

    # Median normalise the columns
    hbo_df[hbo_cols + hb_cols] -= hbo_df[hbo_cols + hb_cols].median()

    # Combine the datasets
    dataset = snirf.NirscordDataset(subject_id, hbo_df, events_df)

    # Write to SNIRF file
    snirf.to_snirf(dataset, f"../data/processed/participant{subject_id}-artinis.snirf")
    valid = validateSnirf(f"../data/processed/participant{subject_id}-mendi.snirf")

    # Cache the dataset for analysis
    if not valid:
        raise Exception("Invalid snirf file")
    else:
        datasets.append(dataset)


In [29]:
import plotly.graph_objects as go


colours = ["red", "orange", "yellow", "green", "blue", "purple", "brown", "pink"]

for dataset in datasets:
    fig = go.Figure()

    #
    time = np.arange(dataset.stream_df.iloc[-1]["timestamp"])

    # --- left channel
    fig_left = go.Figure()

    for i, name in enumerate(MAPPINGS.keys()):
        fig_left.add_trace(go.Scatter(
            x=time, y=dataset.stream_df[f"{name} hbo"],
            mode='lines', name=f'{name} Δ[HbO]',
            line=dict(color=colours[i]),
            hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
        ))

    fig_left.update_layout(
        title=f'Participant {dataset.subject_id} HbO concentrations',
        xaxis_title='Time (s)',
        yaxis_title='Δ Concentration (μM)',
        hovermode='x unified'
    )
    fig_left.show()

# Convert to SNIRF

# Convert to HbO and Hb Concentrations

In [30]:
# Generate a set of time values in the epoch range to display
time = np.arange(-30, 10)

# Collect all SART (No) error epochs into two separate lists
no_error_epochs, error_epochs = utils.collect_epochs(datasets, range(-30, 10))

# ---
error_group = pd.concat(error_epochs).groupby(level=0)
avg_sart_errors = error_group.mean().dropna()
sem_sart_errors = error_group.sem().dropna()

no_error_group = pd.concat(no_error_epochs).groupby(level=0)
avg_sart_no_errors = no_error_group.mean().dropna()
sem_sart_no_errors = no_error_group.sem().dropna()

for i, name in enumerate(MAPPINGS.keys()):
    fig = go.Figure()

    # Calculate the error mean and uncertainty
    error_mean = avg_sart_errors[f"{name} hbo"]
    error_sem = sem_sart_errors[f"{name} hbo"]

    upper_error = error_mean + error_sem
    lower_error = error_mean - error_sem

    # Display the error mean line
    fig.add_trace(go.Scatter(
        x=time,
        y=error_mean,
        mode='lines',
        name='SART Error',
        line=dict(color='rgb(0, 0, 255)'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Display the error sem area
    fig.add_trace(go.Scatter(
        x=time,
        y=lower_error,
        mode='lines',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig.add_trace(go.Scatter(
        x=time,
        y=upper_error,
        mode='lines',
        fill='tonexty',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        fillcolor='rgba(0, 0, 255, 0.2)',
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Calculate the no error mean and uncertainty
    no_error_mean = avg_sart_no_errors[f"{name} hbo"]
    no_error_sem = sem_sart_no_errors[f"{name} hbo"]

    upper_no_error = no_error_mean + no_error_sem
    lower_no_error = no_error_mean - no_error_sem

    # Display the no error mean line
    fig.add_trace(go.Scatter(
        x=time,
        y=no_error_mean,
        mode='lines',
        name='SART No Error',
        line=dict(color='rgb(0, 255, 0)'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Display the no error sem area
    fig.add_trace(go.Scatter(
        x=time,
        y=lower_no_error,
        mode='lines',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig.add_trace(go.Scatter(
        x=time,
        y=upper_no_error,
        mode='lines',
        fill='tonexty',
        showlegend=False,
        line=dict(color="rgba(0,0,0,0)"),
        fillcolor='rgba(0, 255, 0, 0.2)',
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))

    # Add title and metadata to plot
    fig.update_layout(
        title=f'{name.upper()} Average SART (No) Error HbO Concentrations',
        xaxis_title='Time (s)',
        yaxis_title='Δ Concentration (μM)',
        hovermode='x unified'
    )

    fig.show()


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score


def lda_validate(X: pd.DataFrame, Y: pd.DataFrame):
    X_train, X_test, Y_train, Y_test = train_test_split(
        X,  # The average epoch HbO concentrations
        Y,  # Corresponding SART (No) Error labels
        test_size=0.2,
        stratify=Y,
        random_state=42
    )

    # Define the LDA processing and training pipeline
    lda = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lda", LinearDiscriminantAnalysis())
    ])

    # Define the k-fold cross validation configuration
    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Perform k-fold cross validation on the pipeline
    cv_scores = cross_val_score(
        lda,
        X_train,
        Y_train,
        cv=cv,
        scoring="accuracy"
    )

    # Compute both the mean accuracy and standard error
    return cv_scores.mean(), sem(cv_scores)

accuracies = []
std_errors = []

for dataset in datasets:
    no_error_epochs, error_epochs = utils.extract_epochs(
        dataset.stream_df, dataset.events_df, range(-15, -5)
    )

    # Sometimes empty rows occur when recording was stopped
    no_error_epoch_means = pd.DataFrame(
        [epoch.mean() for epoch in no_error_epochs]
    ).dropna(how="all")

    error_epoch_means = pd.DataFrame(
        [epoch.mean() for epoch in error_epochs]
    ).dropna(how="all")

    X = pd.concat([no_error_epoch_means, error_epoch_means])

    X = X.filter(like="hbo")
    X = X.dropna(axis=1, how="all")

    if len(no_error_epoch_means) < 10:
        print(
            f"Subject {dataset.subject_id} | "
            f"Skipped: class no error has only {len(no_error_epoch_means)} samples (< 10)"
        )
        continue

    elif len(error_epoch_means) < 10:
        print(
            f"Subject {dataset.subject_id} | "
            f"Skipped: class no error has only {len(error_epoch_means)} samples (< 10)"
        )
        continue

    Y = np.concatenate([
        np.zeros(len(no_error_epoch_means), dtype=int),
        np.ones(len(error_epoch_means), dtype=int)
    ])

    mean_acc, std_err = lda_validate(X, Y)

    print(
        f"Subject {dataset.subject_id} |",
        f"CV Accuracy: {mean_acc:.1%} ± {std_err:.1%} SE"
    )

    accuracies.append(mean_acc)
    std_errors.append(std_err)

mean_accuracy = np.mean(accuracies)
mean_std_error = np.mean(std_errors)

print("=" * 40)
print(f"Mean Accuracy: {mean_accuracy:.1%}")
print(f"Mean Standard Error: {mean_std_error:.1%}")
print("=" * 40)



Subject 1 | CV Accuracy: 62.5% ± 7.6% SE
Subject 2 | Skipped: class no error has only 5 samples (< 10)
Subject 3 | CV Accuracy: 65.8% ± 9.3% SE
Subject 4 | CV Accuracy: 46.7% ± 7.5% SE
Subject 5 | CV Accuracy: 58.3% ± 6.7% SE
Subject 6 | CV Accuracy: 60.0% ± 9.8% SE
Subject 7 | CV Accuracy: 45.8% ± 9.1% SE
Subject 8 | CV Accuracy: 68.3% ± 3.2% SE
Mean Accuracy: 58.2%
Mean Standard Error: 7.6%


In [32]:
import plotly.express as px

# Collect all SART (No) error epochs into two separate lists
no_error_epochs, error_epochs = utils.collect_epochs(datasets, range(-15, -5))

# Sometimes empty rows occur when recording was stopped
no_error_epoch_means = pd.DataFrame(
    [epoch.mean() for epoch in no_error_epochs]
).dropna(how="all")

error_epoch_means = pd.DataFrame(
    [epoch.mean() for epoch in error_epochs]
).dropna(how="all")

X = pd.concat([no_error_epoch_means, error_epoch_means])

X = X.filter(like="hbo")
X = X.dropna(axis=1, how="all")

Y = np.concatenate([
    np.zeros(len(no_error_epoch_means), dtype=int),
    np.ones(len(error_epoch_means), dtype=int)
])

X_train, X_test, Y_train, Y_test = train_test_split(
    X,  # The average epoch HbO concentrations
    Y,  # Corresponding SART (No) Error labels
    test_size=0.2,
    stratify=Y,
    random_state=42
)

# Define the LDA processing and training pipeline
lda = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis())
])

lda.fit(X_train, Y_train)

Y_pred = lda.predict(X_test)

# Confusion matrix
cm = confusion_matrix(Y_test, Y_pred)

fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale="Blues",
    labels={
        "x": "Predicted",
        "y": "Actual",
        "color": "Count"
    },
    x=["No Error", "Error"],
    y=["No Error", "Error"]
)

fig.update_layout(
    title="LDA Confusion Matrix",
    width=600,
    height=500
)

fig.show()
